# x402 Batch-Settlement — Buyer / Facilitator Spike (Deno / TypeScript)

Phase 0 harness for the batch-settlement migration (see `assistent_plan.md` §F). It drives the
**batch-settlement** scheme against a locally-running facilitator, mirroring `x402_facilitator_demo_ts.ipynb`
(exact scheme). Use it to learn the buyer/facilitator API surface before wiring the scw_js server (Phase B).

**Terminology — x402 SDK role → this repo's package** (the SDK says "client"/"server", which reads ambiguously here):

| SDK role | Repo package | Role in this repo |
|---|---|---|
| **Client** (buyer / payer) | `website/` | Browser wallet: signs the deposit + per-request vouchers |
| **Server** (seller / merchant) | `scw_js/` | Verifies vouchers, serves the LLM response, claims — **not built yet (Phase B)** |
| **Facilitator** | `x402_facilitator/` *(this package)* | Neither buyer nor seller — settles on-chain: verify signatures, submit deposit/claim/settle txs |

This notebook plays the **buyer** role directly against the facilitator — the section headings below say
"buyer", not "client", to keep that unambiguous. Since scw_js's seller/resource-server side doesn't exist
yet, this is a direct buyer→facilitator spike, skipping the seller hop that Phase B will add.

**Prerequisites**
- Deno Jupyter kernel (same as the sibling `*_ts.ipynb` notebooks).
- The package's single **`x402_facilitator/.env`** (one level up — there is no per-notebook `.env`) with:
  `TEST_WALLET_PRIVATE_KEY` (a wallet funded on **Base Sepolia**: ETH for gas + testnet USDC via
  https://faucet.circle.com/), `NFT_WALLET_PUBLIC_KEY` (recipient), and `FACILITATOR_WALLET_PUBLIC_KEY`.
- A facilitator running locally: from the parent dir `npm install && npm run build && npm run dev` → `http://localhost:8080`.

**Why Base Sepolia:** the canonical `BATCH_SETTLEMENT_ADDRESS` (`0x4020…0003`) is deployed on Base Sepolia,
Optimism mainnet and Base mainnet — but **not** Optimism Sepolia. So testnet spikes use Base Sepolia.

> ⚠️ Cells that construct the deposit/voucher payload and hit `/settle` perform **real on-chain** actions
> and use batch-settlement client APIs that should be confirmed at runtime — they are the point of the spike.


In [11]:
// Setup: imports + config
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount, generatePrivateKey } from "npm:viem@2/accounts";
import { createPublicClient, http, formatUnits } from "npm:viem@2";
import { baseSepolia, base, optimism } from "npm:viem@2/chains";

// All env lives in the package's single .env, one level up: x402_facilitator/.env
// (TEST_WALLET_PRIVATE_KEY, NFT_WALLET_PUBLIC_KEY, FACILITATOR_WALLET_*).
// `examplePath: null` disables dotenv's default assertion that every key in a
// ./.env.example must also be present — we're not using a per-notebook .env.
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
if (!PRIVATE_KEY) {
    throw new Error("TEST_WALLET_PRIVATE_KEY missing from x402_facilitator/.env — " +
        "add a key for a wallet funded on the selected network.");
}
const PAY_TO_ADDRESS = env.NFT_WALLET_PUBLIC_KEY ?? "0x209693Bc6afc0C5328bA36FaF03C514EF312287C";
const FACILITATOR_ADDRESS = env.FACILITATOR_WALLET_PUBLIC_KEY; // from ../.env — used for the gas check
const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);

// The receiver self-manages an authorizer key (signs claims/refunds). In production
// scw_js holds this and advertises its address in the 402's extra.receiverAuthorizer.
// For the spike we use a throwaway one.
const receiverAuthorizer = privateKeyToAccount(generatePrivateKey());

console.log("🚀 x402 Batch-Settlement Spike (Deno/TS)");
console.log(`   Payer (buyer)     : ${account.address}`);
console.log(`   Recipient         : ${PAY_TO_ADDRESS}`);
console.log(`   receiverAuthorizer: ${receiverAuthorizer.address}`);

🚀 x402 Batch-Settlement Spike (Deno/TS)
   Payer (buyer)     : 0x553179556FC2A39e535D65b921e01fA995E79101
   Recipient         : 0x209693Bc6afc0C5328bA36FaF03C514EF312287C
   receiverAuthorizer: 0x0aa4Fe189977b6f86510298DF5Aa3437B99Becf2


## Network selection

Modeled on `x402_facilitator_demo.ipynb`, but **Optimism Sepolia is disabled** — the canonical
`BATCH_SETTLEMENT_ADDRESS` (`0x4020…0003`) is **not** deployed there. Valid networks (all have the contract):
Base Sepolia (default), Base Mainnet, Optimism Mainnet.

| Chain | Testnet | Mainnet |
|---|---|---|
| Base | `eip155:84532` ✅ | `eip155:8453` |
| Optimism | ~~`eip155:11155420`~~ (no contract) | `eip155:10` |


In [12]:
// ── Network selection ──────────────────────────────────────────────
const USE_MAINNET = false;  // ⚠️ true = REAL MONEY. Keep false for the spike.
const USE_BASE = true;      // Base by default. (Optimism + testnet is unavailable.)

// Guard: Optimism Sepolia has no batch-settlement contract.
if (!USE_BASE && !USE_MAINNET) {
    throw new Error("Optimism Sepolia is unsupported for batch-settlement (no canonical contract). " +
        "Use Base Sepolia (USE_BASE=true) or a mainnet (USE_MAINNET=true).");
}

const NETWORK_CONFIG = {
    "base-testnet": {
        chain: baseSepolia, chainId: 84532, caip2Network: "eip155:84532" as const,
        networkName: "Base Sepolia (Testnet)", usdcName: "USDC",
        usdcAddress: "0x036CbD53842c5426634e7929541eC2318f3dCF7e" as `0x${string}`,
        rpcUrl: "https://sepolia.base.org", explorer: "https://sepolia.basescan.org",
        faucet: "https://faucet.circle.com/",
    },
    "base-mainnet": {
        chain: base, chainId: 8453, caip2Network: "eip155:8453" as const,
        networkName: "Base Mainnet", usdcName: "USD Coin",
        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913" as `0x${string}`,
        rpcUrl: "https://mainnet.base.org", explorer: "https://basescan.org",
        faucet: "Bridge: https://bridge.base.org",
    },
    "optimism-mainnet": {
        chain: optimism, chainId: 10, caip2Network: "eip155:10" as const,
        networkName: "Optimism Mainnet", usdcName: "USD Coin",
        usdcAddress: "0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85" as `0x${string}`,
        rpcUrl: "https://mainnet.optimism.io", explorer: "https://optimistic.etherscan.io",
        faucet: "Bridge: https://app.optimism.io/bridge",
    },
};
const configKey = USE_MAINNET ? (USE_BASE ? "base-mainnet" : "optimism-mainnet") : "base-testnet";
const config = NETWORK_CONFIG[configKey];

// Max per-request price (6-decimal USDC). Channel deposit ≈ depositMultiplier(5) × this.
const MAX_PRICE = "4000"; // $0.004  → deposit ≈ $0.02

const FACILITATOR_URL = "http://localhost:8080"; // or "https://facilitator.fretchen.eu"
const VERIFY_URL = `${FACILITATOR_URL}/verify`;
const SETTLE_URL = `${FACILITATOR_URL}/settle`;
const SUPPORTED_URL = `${FACILITATOR_URL}/supported`;

console.log(USE_MAINNET ? `🚨 REAL MONEY on ${config.networkName}` : `🧪 ${config.networkName}`);
console.log(`   ${config.caip2Network}  •  USDC ${config.usdcName} @ ${config.usdcAddress}`);
console.log(`   Facilitator: ${FACILITATOR_URL}`);

🧪 Base Sepolia (Testnet)
   eip155:84532  •  USDC USDC @ 0x036CbD53842c5426634e7929541eC2318f3dCF7e
   Facilitator: http://localhost:8080


## 1. `/supported` — assert the facilitator advertises `batch-settlement`

This is the core Phase A check: after registering `BatchSettlementEvmScheme` in `facilitator_instance.ts`,
the facilitator's `/supported` should list the `batch-settlement` scheme for each network.


In [13]:
const supported = await (await fetch(SUPPORTED_URL)).json();
const kinds = supported.kinds ?? [];
const batchKinds = kinds.filter((k: any) => k.scheme === "batch-settlement");
const exactKinds = kinds.filter((k: any) => k.scheme === "exact");

console.log(`schemes advertised: exact=${exactKinds.length}, batch-settlement=${batchKinds.length}`);
console.log("batch-settlement kinds:", JSON.stringify(batchKinds, null, 2));

if (batchKinds.some((k: any) => k.network === config.caip2Network)) {
    console.log(`✅ batch-settlement advertised for ${config.caip2Network}`);
} else {
    console.log(`❌ batch-settlement NOT advertised for ${config.caip2Network} — check facilitator registration`);
}
// Note: with no receiverAuthorizer configured (self-managed receiver), `extra.receiverAuthorizer`
// should be absent here. Confirm — this is spike-unknown #1.

schemes advertised: exact=4, batch-settlement=3
batch-settlement kinds: [
  {
    "x402Version": 2,
    "scheme": "batch-settlement",
    "network": "eip155:10"
  },
  {
    "x402Version": 2,
    "scheme": "batch-settlement",
    "network": "eip155:8453"
  },
  {
    "x402Version": 2,
    "scheme": "batch-settlement",
    "network": "eip155:84532"
  }
]
✅ batch-settlement advertised for eip155:84532


## 2. Buyer setup (the role `website/` will play in Phase C)

This constructs the **buyer**-side scheme — SDK class name `BatchSettlementEvmScheme` imported from
`.../batch-settlement/client` (the SDK's "client" = the paying wallet). In production this code belongs in
`website/`, signing with the connected browser wallet; here `account` is the throwaway spike key from cell 1.

Unlike the exact scheme there is **no `registerBatchSettlementEvmScheme` helper** — construct the scheme
and register it manually on the `x402Client`. A real browser buyer would back `storage` with `localStorage`
(so channel state survives reload); here we use in-memory storage since this is a single-run spike.


In [14]:
import { x402Client } from "npm:@x402/fetch@^2.17.0";
// SDK naming: "client" here means the buyer/payer role (→ website/ in production),
// NOT scw_js (which plays the "server"/seller role — see the terminology table above).
import { BatchSettlementEvmScheme, InMemoryClientChannelStorage }
    from "npm:@x402/evm@^2.17.0/batch-settlement/client";

// Reused later (section 8) to sign additional off-chain vouchers on this same channel.
const buyerSigner = { address: account.address, signTypedData: (a: any) => account.signTypedData(a) };
const buyerStorage = new InMemoryClientChannelStorage();
const buyerScheme = new BatchSettlementEvmScheme(buyerSigner, { storage: buyerStorage });

const buyerClient = new x402Client();
buyerClient.register(config.caip2Network, buyerScheme);
console.log("✅ buyer registered for", config.caip2Network, "(payer:", account.address + ")");

✅ buyer registered for eip155:84532 (payer: 0x553179556FC2A39e535D65b921e01fA995E79101)


## 3. Build batch-settlement payment requirements

In production these come from the **server's 402 response** (the scw_js resource server injects these via the
server scheme's `enhancePaymentRequirements`). **Confirmed required in `extra`:** `receiverAuthorizer`
(non-zero — the client throws `Payment requirements must include a non-zero extra.receiverAuthorizer` without it)
and `withdrawDelay`, plus the usual EIP-712 `name`/`version`.


In [15]:
const paymentRequirements = {
    scheme: "batch-settlement",
    network: config.caip2Network,
    amount: MAX_PRICE,
    asset: config.usdcAddress,
    payTo: PAY_TO_ADDRESS,
    maxTimeoutSeconds: 3600,
    extra: {
        name: config.usdcName,
        version: "2",
        receiverAuthorizer: receiverAuthorizer.address, // server's self-managed authorizer in prod
        withdrawDelay: 86400,
    },
};

const paymentRequired = {
    x402Version: 2,
    accepts: [paymentRequirements],
    resource: { url: "https://example.com/llm", description: "batch-settlement spike", mimeType: "application/json" },
    extensions: {},
};
console.log(JSON.stringify(paymentRequirements, null, 2));

{
  "scheme": "batch-settlement",
  "network": "eip155:84532",
  "amount": "4000",
  "asset": "0x036CbD53842c5426634e7929541eC2318f3dCF7e",
  "payTo": "0x209693Bc6afc0C5328bA36FaF03C514EF312287C",
  "maxTimeoutSeconds": 3600,
  "extra": {
    "name": "USDC",
    "version": "2",
    "receiverAuthorizer": "0x0aa4Fe189977b6f86510298DF5Aa3437B99Becf2",
    "withdrawDelay": 86400
  }
}


## 4. Buyer creates the deposit + first cumulative voucher payload

On the first request the buyer opens a channel: an EIP-3009 deposit into the batch-settlement contract plus
an initial cumulative voucher. `createPaymentPayload` handles the signing. **Confirmed payload shape:**
`payload.type == "deposit"` with `channelConfig` (payer, payerAuthorizer, receiver, receiverAuthorizer, token,
withdrawDelay, salt), `voucher` (channelId, maxClaimableAmount, signature), and `deposit`. The deposit is sized
by the deposit policy (default `depositMultiplier: 5`), so the payer needs ≥ 5×MAX_PRICE USDC to settle.


In [16]:
const paymentPayload = await buyerClient.createPaymentPayload(paymentRequired as any);
console.log("✅ payload created (deposit + voucher):");
console.log(JSON.stringify(paymentPayload, null, 2).slice(0, 1200));

✅ payload created (deposit + voucher):
{
  "x402Version": 2,
  "payload": {
    "type": "deposit",
    "channelConfig": {
      "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
      "payerAuthorizer": "0x553179556FC2A39e535D65b921e01fA995E79101",
      "receiver": "0x209693Bc6afc0C5328bA36FaF03C514EF312287C",
      "receiverAuthorizer": "0x0aa4Fe189977b6f86510298DF5Aa3437B99Becf2",
      "token": "0x036CbD53842c5426634e7929541eC2318f3dCF7e",
      "withdrawDelay": 86400,
      "salt": "0x0000000000000000000000000000000000000000000000000000000000000000"
    },
    "voucher": {
      "channelId": "0x18afc0ccce49b2c9171136dca9711e101f635ec3edb6610a6129a87ac7d71986",
      "maxClaimableAmount": "4000",
      "signature": "0xa61dd50170b4bd6c7ba777140d7fe30512ea0cbe3991d00e30c9e68e9759f24736214431b0c5aea5aa4473f6752480e041a94488733fb8ed7a64476d3c559a781b"
    },
    "deposit": {
      "amount": "20000",
      "authorization": {
        "erc3009Authorization": {
          "validAfter":

## 5. `/verify` the payload


In [17]:
const verifyRes = await fetch(VERIFY_URL, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ paymentPayload, paymentRequirements }),
});
const verifyResult = await verifyRes.json();
console.log(`verify (${verifyRes.status}):`, JSON.stringify(verifyResult, null, 2));

verify (200): {
  "isValid": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101"
}


## 6. Pre-settlement balance check (like the demo notebook)

Settlement escrows the deposit on-chain, so the **payer needs USDC** (≈ `depositMultiplier` × MAX_PRICE) and
the **facilitator wallet needs ETH** for gas. Check before spending.


In [18]:
const publicClient = createPublicClient({ chain: config.chain, transport: http(config.rpcUrl) });
const erc20 = [{ inputs: [{ name: "a", type: "address" }], name: "balanceOf",
    outputs: [{ name: "", type: "uint256" }], stateMutability: "view", type: "function" }] as const;

const payerUsdc = await publicClient.readContract({
    address: config.usdcAddress, abi: erc20, functionName: "balanceOf", args: [account.address] });
const DEPOSIT_MULTIPLIER = 5n; // batch-settlement default
const needed = BigInt(MAX_PRICE) * DEPOSIT_MULTIPLIER;

console.log(`💵 Payer USDC: ${formatUnits(payerUsdc, 6)}  (deposit needs ≈ ${formatUnits(needed, 6)})`);
if (payerUsdc < needed) {
    console.log(`   ⚠️ insufficient USDC on ${config.networkName} — ${config.faucet}`);
} else {
    console.log(`   ✅ enough USDC to open the channel`);
}

// The facilitator submits the deposit tx, so it needs ETH for gas (from ../.env).
if (FACILITATOR_ADDRESS) {
    const facEth = await publicClient.getBalance({ address: FACILITATOR_ADDRESS as `0x${string}` });
    console.log(`⛽ Facilitator ETH: ${formatUnits(facEth, 18)}  (${FACILITATOR_ADDRESS})`);
    if (facEth === 0n) console.log(`   ⚠️ facilitator has no ETH for gas on ${config.networkName}`);
} else {
    console.log("⛽ FACILITATOR_WALLET_PUBLIC_KEY not found in ../.env — skipping gas check");
}

💵 Payer USDC: 19.92  (deposit needs ≈ 0.02)
   ✅ enough USDC to open the channel
⛽ Facilitator ETH: 0.009996479580891215  (0x3F8d2Fb6fEA24E70155bC61471936F3c9C30c206)


## 7. `/settle` — submit the deposit on-chain

> ⚠️ Real on-chain tx: escrows USDC into the batch-settlement contract. The facilitator dispatches the deposit
> internally by payload type — no separate endpoint. batch-settlement is **fee-free** (no fee `transferFrom`).


In [19]:
if (USE_MAINNET) throw new Error("Refusing to auto-settle on mainnet. Set USE_MAINNET=false or run this cell deliberately.");

const settleRes = await fetch(SETTLE_URL, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ paymentPayload, paymentRequirements }),
});
const settleResult = await settleRes.json();
console.log(`settle (${settleRes.status}):`, JSON.stringify(settleResult, null, 2));
if (settleResult.transaction) console.log(`🔗 ${config.explorer}/tx/${settleResult.transaction}`);
// Fee-free: settleResult should carry NO facilitatorFees / fee.collected.

settle (200): {
  "success": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "transaction": "0x9f7c8794a1d8944963880566869df96d5ecf68372418d078d50978c22b32159a",
  "network": "eip155:84532"
}
🔗 https://sepolia.basescan.org/tx/0x9f7c8794a1d8944963880566869df96d5ecf68372418d078d50978c22b32159a


## 8. Accumulate several requests in the same channel (off-chain, no on-chain action)

This is the actual batching mechanic: once a channel is open (section 4), each further "request" — a chat
message, an image generation, anything metered per-call — just signs a **new, larger cumulative** voucher for
the SAME `channelId`. No transaction, no gas, no facilitator round-trip required. `signVoucher` is a standalone
buyer-side helper (same `.../batch-settlement/client` subpath) for exactly this — it doesn't re-open the channel,
it just bumps the authorized max.


In [20]:
import { signVoucher } from "npm:@x402/evm@^2.17.0/batch-settlement/client";

const channelId = paymentPayload.payload.voucher.channelId;
const channelConfig = paymentPayload.payload.channelConfig;

// "Request #2" and "#3" on the SAME channel — purely off-chain signing, cumulative amount grows.
const voucher2 = await signVoucher(buyerSigner, channelId, "8000", config.caip2Network);
console.log("request #2 -> cumulative maxClaimable:", voucher2.maxClaimableAmount);

const voucher3 = await signVoucher(buyerSigner, channelId, "12000", config.caip2Network);
console.log("request #3 -> cumulative maxClaimable:", voucher3.maxClaimableAmount);
console.log("\n✅ 3 requests accumulated in one channel, zero on-chain transactions so far.");

request #2 -> cumulative maxClaimable: 8000
request #3 -> cumulative maxClaimable: 12000

✅ 3 requests accumulated in one channel, zero on-chain transactions so far.


## 9. Claim the accumulated amount in ONE on-chain transaction

This is the batch-settlement payoff: settle 3 accumulated off-chain requests with a single claim, instead of
3 separate on-chain payments. A **claim** is an authoritative settlement command signed by the
`receiverAuthorizer` (the entity that would run the claim/settle cron in Phase B — here, this spike notebook
stands in for that role, since it already holds that throwaway key from cell 1).

**Important finding from this spike:** the SDK's own `scheme.verify()` has no branch for `"claim"` payloads at
all (only deposit/voucher/refund are verifiable — a claim is a command, not a future payment to verify).
`x402_facilitator/x402_settle.ts::settlePayment()` originally called `verifyPayment()` unconditionally before
`settle()`, which made every claim fail with `invalid_batch_settlement_evm_payload_type` before ever reaching
the scheme's real claim-handling logic. **Fixed** — `settlePayment()` now skips the verify-first gate for
`claim`/`settle`-type batch-settlement payloads and calls `facilitator.settle()` directly (see `x402_settle.ts`
and its regression tests in `test/x402_settle.test.js`).

`claimBatchTypes` and `BATCH_SETTLEMENT_DOMAIN`/`BATCH_SETTLEMENT_ADDRESS` (used to build the same EIP-712
domain the SDK signs internally) are public root exports of `@x402/evm` — no private/internal API needed.

> ⚠️ Real on-chain tx. Needs the section-7 deposit to have actually settled on Base Sepolia first (needs a
> funded wallet) for the claim to move real funds — otherwise it succeeds with **zero effect** (see below).


In [21]:
import { claimBatchTypes, BATCH_SETTLEMENT_DOMAIN, BATCH_SETTLEMENT_ADDRESS } from "npm:@x402/evm@^2.17.0";
import { getAddress } from "npm:viem@2";

if (USE_MAINNET) throw new Error("Refusing to auto-claim on mainnet. Set USE_MAINNET=false or run this cell deliberately.");

// The receiver (server) authorizes claiming this channel's latest voucher. In
// production the server (scw_js) signs this with its own receiverAuthorizer key;
// here we use the throwaway one generated in cell 1.
const claimEip712Domain = {
    ...BATCH_SETTLEMENT_DOMAIN, chainId: config.chainId,
    verifyingContract: getAddress(BATCH_SETTLEMENT_ADDRESS),
};
const claimAuthorizerSignature = await receiverAuthorizer.signTypedData({
    domain: claimEip712Domain, types: claimBatchTypes, primaryType: "ClaimBatch",
    message: { claims: [{
        channelId, maxClaimableAmount: BigInt(voucher3.maxClaimableAmount), totalClaimed: 0n,
    }] },
});

const claimPayload = {
    type: "claim",
    claims: [{
        voucher: { channel: channelConfig, maxClaimableAmount: voucher3.maxClaimableAmount },
        signature: voucher3.signature, totalClaimed: "0",
    }],
    claimAuthorizerSignature,
};
const claimSettleRes = await fetch(SETTLE_URL, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({
        paymentPayload: { x402Version: 2, accepted: paymentRequirements, payload: claimPayload },
        paymentRequirements,
    }),
});
const claimResult = await claimSettleRes.json();
console.log(`claim /settle (${claimSettleRes.status}):`, JSON.stringify(claimResult, null, 2));
if (claimResult.transaction) {
    console.log(`🔗 ${config.explorer}/tx/${claimResult.transaction}`);
    console.log("\nNote: if section 7's deposit was never actually funded/settled, this tx succeeds");
    console.log("on-chain (status 1) but moves ZERO real value and emits no events — check the explorer's");
    console.log("Logs tab. That's the contract correctly no-op'ing an empty channel rather than reverting");
    console.log("the whole (potentially multi-channel) claim batch.");
}

claim /settle (200): {
  "success": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "transaction": "0x1190024fe1a9002f0683e87ddc21fe43cf6b7f95195dff716633855c8a69bb23",
  "network": "eip155:84532"
}
🔗 https://sepolia.basescan.org/tx/0x1190024fe1a9002f0683e87ddc21fe43cf6b7f95195dff716633855c8a69bb23

Note: if section 7's deposit was never actually funded/settled, this tx succeeds
on-chain (status 1) but moves ZERO real value and emits no events — check the explorer's
Logs tab. That's the contract correctly no-op'ing an empty channel rather than reverting
the whole (potentially multi-channel) claim batch.


## Spike findings (confirmed) + what feeds Phase B

- ✅ **Facilitator advertises batch-settlement** for all networks via `/supported`, with **no** `receiverAuthorizer`
  in `extra` → confirms self-managed receiver (scw_js holds the authorizer key, not the facilitator).
- ✅ **Requirements need `extra.receiverAuthorizer` (non-zero) + `withdrawDelay`** — the client throws without them.
- ✅ **Deposit payload shape**: `type="deposit"` + channelConfig + voucher (channelId/maxClaimableAmount/signature) + deposit.
- ✅ **Verify wiring works**: an unfunded payer returns `invalid_batch_settlement_evm_insufficient_balance` — i.e.
  signature/domain/channel config are all accepted, only the balance stops it. (Automated: see
  `test/integration/x402_batch_settlement.integration.test.js`, run via `npm run test:integration`.)
- ✅ **Channel accumulation confirmed** (section 8): `signVoucher` bumps the cumulative amount for an existing
  channel purely off-chain (no tx) — this is the actual "batch several requests" mechanic.
- 🐛→✅ **Found and fixed a real facilitator bug** (section 9): `scheme.verify()` has no branch for `"claim"`
  payloads (by SDK design — a claim is a settlement command, not a future payment to verify), but
  `x402_settle.ts::settlePayment()` unconditionally verified-then-settled everything, so every claim failed
  with `invalid_batch_settlement_evm_payload_type` before reaching the scheme's real claim logic. Fixed by
  skipping the verify gate for `claim`/`settle`-type batch-settlement payloads. Confirmed against a real,
  successfully-mined Base Sepolia transaction (verified via `eth_getTransactionReceipt`: `status: "0x1"`,
  `to` = `BATCH_SETTLEMENT_ADDRESS`) — moved zero real value since the underlying channel wasn't actually
  funded in this spike run, but proves the full wiring (types, signatures, dispatch, RPC submission) end to end.

**Still to exercise here (needs a funded Base Sepolia payer):** run section 7's deposit `/settle` for real, THEN
section 9's claim — that combination proves actual USDC moves on a single aggregated claim. Also: refund.
Deposit = `depositMultiplier` (5) × MAX_PRICE, so fund the payer accordingly.

**Phase B next:** wire the scw_js server scheme (`BatchSettlementEvmScheme(receiver, { storage: s3Storage })`) + the
S3 `ChannelStorage`, use `setSettlementOverrides` to claim the actual post-generation cost, then the claim/settle
cron (which is exactly the `claim`-payload flow this spike just proved out — Phase B needs the receiverAuthorizer
key + BatchSettlementChannelManager, not hand-rolled EIP-712 signing). See `assistent_plan.md` §F.
